# Grad-CAM for the final transfer-learning models

This notebook requires the four checkpoints and saved validation-selected operating thresholds produced by the current comparison run. It uses only validation images for these qualitative examples, architecture-specific DenseNet121 or ResNet50 preprocessing, and the 90% validation-sensitivity thresholds used in evaluation. It generates correctly classified NORMAL and PNEUMONIA examples plus an incorrect example for each model when those cases exist. Grad-CAM is a qualitative aid, not proof of clinical reasoning or of an artifact causing a prediction.


In [ ]:
# Set SPLIT_DIR below if the prepared data is not in the default KaggleHub cache.
from pathlib import Path
import matplotlib.pyplot as plt
import json
import subprocess
import sys
import numpy as np
import tensorflow as tf

PROJECT_DIR = next(
    (parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (parent / "requirements.txt").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Start Jupyter from this project or set PROJECT_DIR explicitly.")
SPLIT_DIR = None
if SPLIT_DIR is None:
    versions = (Path.home() / ".cache" / "kagglehub" / "datasets" /
                "paultimothymooney" / "chest-xray-pneumonia" / "versions")
    candidates = [PROJECT_DIR / "data" / "chest_xray_cnn_splits",
                  PROJECT_DIR / "chest_xray_cnn_splits",
                  versions / "chest_xray_cnn_splits",
                  *sorted(versions.glob("*/chest_xray_cnn_splits"))]
    SPLIT_DIR = next((p for p in candidates if (p / "val").is_dir()), None)
if SPLIT_DIR is None:
    raise FileNotFoundError("Set SPLIT_DIR to the prepared split containing val/NORMAL and val/PNEUMONIA.")
SPLIT_DIR = Path(SPLIT_DIR).resolve()
VALIDATION_DIR = SPLIT_DIR / "val"
OUTPUT_DIR = PROJECT_DIR / "results" / "gradcam"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
RUN_ID = "seed42_run01"
USE_EXISTING_FROZEN_CHECKPOINTS = False
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
TARGET_SENSITIVITY = 0.90
MANIFEST_SUMMARY = json.loads((PROJECT_DIR / "data" / "dataset_manifest_summary.json").read_text(encoding="utf-8"))
MANIFEST_SHA256 = MANIFEST_SUMMARY["manifest_sha256"]

frozen_checkpoint_paths = (
    {
        "DenseNet121": PROJECT_DIR / "models" / "frozen" / "DenseNet121_frozen.keras",
        "ResNet50": PROJECT_DIR / "models" / "frozen" / "ResNet50_frozen.keras",
    }
    if USE_EXISTING_FROZEN_CHECKPOINTS
    else {
        "DenseNet121": PROJECT_DIR / "models" / "frozen" / f"DenseNet121_{RUN_ID}_frozen.keras",
        "ResNet50": PROJECT_DIR / "models" / "frozen" / f"ResNet50_{RUN_ID}_frozen.keras",
    }
)
CHECKPOINTS = {
    "frozen": frozen_checkpoint_paths,
    "fine_tuned": {
        "DenseNet121": PROJECT_DIR / "models" / "fine_tuned" / f"DenseNet121_{RUN_ID}_fine_tuned.keras",
        "ResNet50": PROJECT_DIR / "models" / "fine_tuned" / f"ResNet50_{RUN_ID}_fine_tuned.keras",
    },
}
print("Validation folder:", VALIDATION_DIR)
print("TensorFlow:", tf.__version__)


## Load validation images and the four transfer checkpoints

Use the prepared validation set in fixed class order. Missing checkpoint files are listed; available checkpoints are checked for expected image and output shapes.

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_DIR / "src" / "data" / "validate_dataset.py"), "--split-dir", str(SPLIT_DIR)],
    check=True,
)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    VALIDATION_DIR,
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)
images = np.concatenate([x.numpy() for x, _ in validation_ds])
labels = np.concatenate([y.numpy() for _, y in validation_ds])
if images.shape[1:] != (*IMAGE_SIZE, 3):
    raise ValueError(f"Unexpected validation image shape: {images.shape}")
print("Class mapping:", {name: index for index, name in enumerate(CLASS_NAMES)})
print("Validation counts:", dict(zip(CLASS_NAMES, np.bincount(labels, minlength=2))))

preprocess_functions = {
    "DenseNet121": tf.keras.applications.densenet.preprocess_input,
    "ResNet50": tf.keras.applications.resnet50.preprocess_input,
}
models = {}
model_strategies = {}
model_thresholds = {}
for strategy, checkpoints in CHECKPOINTS.items():
    for architecture, path in checkpoints.items():
        model_name = f"{architecture}_{strategy.title().replace('_', '')}"
        if not path.is_file():
            raise FileNotFoundError(f"Required final checkpoint is missing for {model_name}: {path}")
        metrics_path = PROJECT_DIR / "results" / "transfer_learning" / RUN_ID / "per_model" / model_name / "metrics.json"
        if not metrics_path.is_file():
            raise FileNotFoundError(f"Validation-selected threshold record is missing: {metrics_path}")
        record = json.loads(metrics_path.read_text(encoding="utf-8"))
        if record.get("run_id") != RUN_ID or record.get("manifest_sha256") != MANIFEST_SHA256:
            raise ValueError(f"Metrics provenance does not match the selected run and dataset: {metrics_path}")
        model = tf.keras.models.load_model(
            path,
            compile=False,
            safe_mode=True,
            custom_objects={"preprocess_input": preprocess_functions[architecture]},
        )
        if model.input_shape != (None, 224, 224, 3) or model.output_shape != (None, 2):
            raise ValueError(f"{model_name}: incompatible input/output shapes: {model.input_shape}, {model.output_shape}")
        backbone = model.get_layer(architecture.lower())
        if strategy == "frozen" and (backbone.trainable or backbone.trainable_weights):
            raise ValueError(f"{model_name}: expected a frozen backbone.")
        models[model_name] = model
        model_strategies[model_name] = (strategy, architecture)
        model_thresholds[model_name] = float(record["threshold"])
        print(
            f"{model_name}: loaded {path} | backbone trainable={backbone.trainable} | "
            f"validation threshold={model_thresholds[model_name]:.4f}"
        )


## Grad-CAM

The target is the final convolutional layer within the selected DenseNet121 or ResNet50 backbone. The saved model's augmentation and matching preprocessing layers are included in the forward pass.

In [ ]:
def gradcam(model, image, class_index):
    backbone = next(layer for layer in model.layers if layer.name in ('densenet121', 'resnet50'))
    last_conv = next(layer for layer in reversed(backbone.layers)
                     if isinstance(layer, tf.keras.layers.Conv2D))
    cam_backbone = tf.keras.Model(backbone.input, [last_conv.output, backbone.output])
    image = tf.convert_to_tensor(image[None, ...], dtype=tf.float32)
    augmentation = model.get_layer('training_augmentation')
    preprocess = model.get_layer('preprocess')
    prepared = preprocess(augmentation(image, training=False))
    backbone_index = model.layers.index(backbone)
    with tf.GradientTape() as tape:
        features, values = cam_backbone(prepared, training=False)
        for layer in model.layers[backbone_index + 1:]:
            values = layer(values, training=False)
        score = values[:, class_index]
    gradients = tape.gradient(score, features)
    weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    heatmap = tf.nn.relu(tf.reduce_sum(features * weights, axis=-1))[0]
    heatmap = tf.image.resize(heatmap[..., None], IMAGE_SIZE)[..., 0]
    return (heatmap / (tf.reduce_max(heatmap) + 1e-8)).numpy()

## Compare the same validation X-rays

Choose the first correctly and incorrectly classified validation examples for DenseNet121 Frozen when available, then display those same examples for every loaded model. Figures are separated by frozen/fine-tuned strategy and architecture.

In [ ]:
if not models:
    raise RuntimeError("No final transfer-learning checkpoints were loaded.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, model in models.items():
    strategy, architecture = model_strategies[name]
    scores = model.predict(images, batch_size=BATCH_SIZE, verbose=0)
    pneumonia_predictions = (scores[:, 1] >= model_thresholds[name]).astype(np.int32)
    example_groups = (
        ("correct_NORMAL", np.flatnonzero((labels == 0) & (pneumonia_predictions == 0))),
        ("correct_PNEUMONIA", np.flatnonzero((labels == 1) & (pneumonia_predictions == 1))),
        ("incorrect", np.flatnonzero(labels != pneumonia_predictions)),
    )
    for example_type, candidate_indices in example_groups:
        if not candidate_indices.size:
            print(f"{name}: no validation example available for {example_type}.")
            continue
        index = int(candidate_indices[0])
        predicted = int(pneumonia_predictions[index])
        confidence = float(scores[index, predicted])
        heatmap = gradcam(model, images[index], predicted)
        figure, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(images[index].astype("uint8"))
        axes[0].set_title("Original X-ray")
        axes[1].imshow(images[index].astype("uint8"))
        axes[1].imshow(heatmap, cmap="jet", alpha=0.45, vmin=0, vmax=1)
        axes[1].set_title("Heatmap overlay")
        axes[2].imshow(heatmap, cmap="jet", vmin=0, vmax=1)
        axes[2].set_title("Grad-CAM heatmap")
        for axis in axes:
            axis.axis("off")
        figure.suptitle(
            f"{name} | {example_type} | actual: {CLASS_NAMES[int(labels[index])]} | "
            f"predicted: {CLASS_NAMES[predicted]} ({confidence:.3f}) | "
            f"validation threshold={model_thresholds[name]:.3f} | index={index}"
        )
        figure.tight_layout()
        output_path = OUTPUT_DIR / strategy / architecture / f"{RUN_ID}_{example_type}_{index}.png"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        if output_path.exists():
            raise FileExistsError(f"Refusing to overwrite an existing Grad-CAM figure: {output_path}")
        figure.savefig(output_path, dpi=140)
        plt.show()
        plt.close(figure)
        print("Saved:", output_path)
print("Grad-CAM figures are qualitative only; review borders, markers, and non-lung regions manually.")


## Visual review and limitations

Compare each heatmap with the original image: note activation within the thoracic/lung field and any attention on borders, corners, text, or markers. The notebook has no lung masks; these are qualitative observations only and do not establish that an artifact caused a prediction or that a model reasons clinically.

The preparation code stratifies images but does not group by patient, so patient-level leakage cannot be ruled out. The dataset may also contain label noise and source-specific bias. Grad-CAM is not clinical validation; this academic prototype is not a diagnostic tool. Validation images are used for explanation only; do not tune using the test set.